# 8.4 低秩分解 (Low-Rank Decomposition)

低秩分解将大权重矩阵分解为两个小矩阵的乘积，减少参数量和计算量。

**目的**：通过矩阵分解减少参数

**基本原理**：利用权重矩阵的低秩特性，将W∈R^(m×n)分解为U∈R^(m×r)和V∈R^(r×n)，其中r远小于min(m,n)。

**核心公式**：
- W ≈ U × V
- 参数量：mn → (m+n)r，当r << min(m,n)时大幅减少
- 计算量：O(mn) → O((m+n)r)

**与LoRA的区别**：
- 低秩分解：替换原始权重，减少参数
- LoRA：在原始权重旁添加低秩增量，不减少参数

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

W = torch.randn(1024, 1024)

U, S, Vh = torch.linalg.svd(W, full_matrices=False)

for rank in [32, 64, 128, 256]:
    W_approx = U[:, :rank] @ torch.diag(S[:rank]) @ Vh[:rank, :]
    error = (W - W_approx).norm() / W.norm()
    orig_params = 1024 * 1024
    decomposed_params = 1024 * rank * 2
    compression = orig_params / decomposed_params
    print(f'Rank {rank:>3d}: error={error:.4f}, compression={compression:.1f}x, params={decomposed_params:,}')

class LowRankLinear(nn.Module):
    def __init__(self, in_features, out_features, rank=64):
        super().__init__()
        self.U = nn.Linear(in_features, rank, bias=False)
        self.V = nn.Linear(rank, out_features, bias=False)

    def forward(self, x):
        return self.V(self.U(x))

full_linear = nn.Linear(1024, 1024, bias=False)
lowrank_linear = LowRankLinear(1024, 1024, rank=64)

full_params = sum(p.numel() for p in full_linear.parameters())
lr_params = sum(p.numel() for p in lowrank_linear.parameters())

x = torch.randn(4, 1024)
out = lowrank_linear(x)

print(f'\n=== Low-Rank Decomposition ===')
print(f'Full linear: {full_params:,} params')
print(f'Low-rank (r=64): {lr_params:,} params ({lr_params/full_params:.1%})')
print(f'Output: {out.shape}')
print(f'\nKey: SVD-based decomposition finds optimal low-rank approximation.')
print(f'Trade-off: lower rank = more compression but more error.')